# Chains

**LCEL(LangChain Expression Language)** 을 사용해서 모든 구성 요소가 `Runnable` 인터페이스로 통합되어 파이프라인(`|`)으로 연결될 수 있다.

```python
chain = prompt | model | output_parser  # 기본 구조
```

**구성 요소 업데이트 (v1.2 기준)**

1. **PromptTemplate**
   - `Runnable`로 변환되어 LCEL 파이프라인에 직접 통합

   ```python
   prompt = ChatPromptTemplate.from_template("...")
   ```

2. **LLM/ChatModel**
   - `ChatOpenAI`, `ChatAnthropic` 등이 `Runnable` 구현

   ```python
   model = ChatOpenAI(model="gpt-4-turbo")
   ```

3. **Output Parsers**
   - `StrOutputParser()`, `JsonOutputParser()` 등이 `Runnable`로 작동

   ```python
   output_parser = JsonOutputParser()
   ```

4. **Tools**
   - `@tool` 데코레이터로 도구 정의

   ```python
   @tool
   def search(query: str) -> str:
       ...
   ```

**체인 유형별 구현**

1. Simple Chain

   ```python
   chain = prompt | model | output_parser
   response = chain.invoke({"input": "..."})
   ```

2. Sequential Chain

   ```python
   chain = (
       {"step1_output": prompt1 | model1}  # 첫 번째 체인 결과 매핑
       | prompt2
       | model2
   )
   ```

3. Conditional Chain
   - `RunnableBranch` 사용

   ```python
   branch = RunnableBranch(
       (lambda x: x["topic"] == "math", math_chain),
       (lambda x: x["topic"] == "history", history_chain),
       default_chain
   )
   ```

**v1.2 주요 변경점**

- **Legacy Chain 클래스**: `LLMChain`, `SequentialChain` 등은 `langchain-classic`으로 이동되거나 삭제됨 → `Runnable`(LCEL)로 통합
- **에이전트 통합**: `create_agent`(LangGraph 기반)가 표준

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_core.prompts import PromptTemplate
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser

llm = init_chat_model("openai:gpt-4.1-mini")
output_parser = StrOutputParser()

## Simple chain

In [3]:
prompt = PromptTemplate.from_template('{country}의 수도는 어디입니까?')
chain = prompt | llm | output_parser
chain.invoke(input={'country':'대한민국'})

'대한민국의 수도는 서울특별시입니다.'

## sequential Chain
- 두 개 이상의 chain을 직렬로 연결
- 번역 chain 과 요약 chain을 연결하는 예제

In [4]:
prompt1 = PromptTemplate.from_template("다음 문장을 한글로 번역하세요.: \n\n{eng_text}")
prompt2 = PromptTemplate.from_template("다음 문장을 한 문장으로 짧게 요약하세요.: \n\n{kor_text}")

# 번역 체인
translation_chain = prompt1 | llm
# 요약 체인
summaery_chain = prompt2 | llm | output_parser

# 통합 체인
chain = translation_chain | summaery_chain

sentence = '''
"AI" redirects here. For other uses, see AI (disambiguation) and Artificial intelligence (disambiguation).

Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.[1]

High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, and play and analysis in strategy games (e.g., chess and Go). Since the 2020s, generative AI has become widely available to generate images, audio, and videos from text prompts.

The traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robotics.[a] To reach these goals, AI researchers have used techniques including state space search and mathematical optimization, formal logic, artificial neural networks, and methods based on statistics, operations research, and economics.[b] AI also draws upon psychology, linguistics, philosophy, neuroscience, and other fields.[2] Some companies, such as OpenAI, Google DeepMind and Meta, aim to create artificial general intelligence (AGI) – AI that can complete virtually any cognitive task at least as well as a human.[3]

Artificial intelligence was founded as an academic discipline in 1956,[4] and the field went through multiple cycles of optimism throughout its history,[5][6] followed by periods of disappointment and loss of funding, known as AI winters.[7][8] Funding and interest increased substantially after 2012, when graphics processing units began being used to accelerate neural networks, and deep learning outperformed previous AI techniques.[9] This growth accelerated further after 2017 with the transformer architecture.[10] In the 2020s, an AI boom has coincided with advances in generative AI, which allowed for the creation and modification of media. In addition to AI safety and unintended consequences and harms from the use of AI, ethical concerns, AI's long-term effects, and potential existential risks have prompted discussions of AI regulation.

Goals
'''

print(chain.invoke(sentence))

인공지능(AI)은 인간 지능과 유사한 학습, 추론, 문제 해결 등을 수행하는 컴퓨터 시스템을 연구하며, 다양한 응용 분야와 기술 발전을 거치면서 사회적·윤리적 이슈와 함께 발전해온 기술입니다.


## Conditional Chain

In [6]:
from langchain_core.runnables import RunnableBranch

# 수학 선생님 체인
math_prompt = PromptTemplate.from_template("다음 문제를 풀어주세요. 단계적인 풀이를 수식(LaTex)와 함께 작성해주세요.\n\n{question}")
math_chain = math_prompt | llm | output_parser

# 기본 체인
default_prompt = PromptTemplate.from_template("당신은 친절하고, 감성적인 공감 능력이 좋은 챗봇입니다. 다음 질문에 답해주세요. \n\n{question}")
default_chain = default_prompt | llm | output_parser

# main_chain 선택 함수, math_chain 을 사용해야할 경우 True 변환
def is_math_question(input_dict : dict) -> bool:
    question:str = input_dict.get("question","")
    return '계산' in question or 'calc' in question

# 분기 체인
branch_chain = RunnableBranch(
    (is_math_question,math_chain),
    default_chain
)

# 수학 질문
print(branch_chain.invoke({'question':'1234*3+30 이거좀 계산해줘'}))

# 그와 질문
print(branch_chain.invoke({'question':'나 오늘 부장님한테 깨졌어. 우울하다ㅠㅠ'}))


물론입니다. 주어진 식을 단계적으로 계산해보겠습니다.

문제:  
\[
1234 \times 3 + 30
\]

1단계: \(1234\)와 \(3\)을 곱합니다.  
\[
1234 \times 3 = 3702
\]

2단계: 곱한 결과에 \(30\)을 더합니다.  
\[
3702 + 30 = 3732
\]

따라서 최종 답은:  
\[
\boxed{3732}
\]
아이고, 정말 속상했겠어요... 부장님한테 그런 말을 들으면 마음이 많이 무거워지고 우울해질 수밖에 없죠. 혹시 어떤 일이 있어서 그런 말씀을 들은 건가요? 괜히 혼자 마음앓이 하지 말고, 조금 더 이야기해줘도 괜찮아요. 당신의 마음을 이해해주고 싶어요.


## 실습: 질문 유형에 따른 다른 제안 실행하기
- 사용자의 입력이 번역 요청인지, 요약 요청인지, 일반 질문인지에 따라 서로 다른 제안을 실행한다.

In [7]:
translation_prompt = PromptTemplate.from_template("다음 문장을 한글로 번역하세요.: \n\n{eng_text}")
translation_chain = translation_prompt | llm | output_parser
summary_prompt = PromptTemplate.from_template("다음 문장을 한 문장으로 짧게 요약하세요.: \n\n{kor_text}")
summary_chain = summary_prompt | llm | output_parser
default_prompt = PromptTemplate.from_template("당신은 친절하고, 감성적인 공감 능력이 좋은 챗봇입니다. 다음 질문에 답해주세요. \n\n{question}")
default_chain = default_prompt | llm | output_parser

In [12]:
def is_translation_question(input_dict : dict) -> bool:
    if isinstance(input_dict, dict):
        question = input_dict.get("eng_text", "")
    else:
        question = input_dict
    return '번역' in question or 'translation' in question

def is_summary_question(input_dict : dict) -> bool:
    if isinstance(input_dict, dict):
        question = input_dict.get("kor_text", "")
    else:
        question = input_dict
    return '요약' in question or 'summary' in question


router_chain = RunnableBranch(
    (is_translation_question,translation_chain),
    (is_summary_question,summary_chain),
    default_chain
)

In [13]:
# 번역 요청
router_chain.invoke("다음 문장으로 한국어로 번역해줘: LangChain helps developers build LLM applicatiions.")

'LangChain은 개발자들이 대형 언어 모델(LLM) 애플리케이션을 구축할 수 있도록 도와줍니다.'

In [14]:
# 요약 요청
router_chain.invoke("""
다음 내용을 요약해줘.
                   
LangChain은 LLM 애플리케이션을 만들기 위한 프레임워크이다.
Prompt, Model, Output Parser, Retriever 등을 조합하여 복잡한 흐름을 구성할 수 있다.
LCEL을 사용하면 각 구성 요소를 파이프라인처럼 연결할 수 있다.                   
""")

'LangChain은 LLM 애플리케이션 개발을 위해 Prompt, Model, Output Parser, Retriever 등의 구성 요소를 파이프라인처럼 연결해 복잡한 흐름을 만드는 프레임워크이다.'

In [15]:
# 일반 질문
router_chain.invoke("LangChain의 LCEL이 뭐야?")

'안녕하세요! LangChain의 LCEL에 대해 궁금해 하시는군요. LCEL은 “LangChain Expression Language”의 약자로, LangChain 내에서 복잡한 논리나 작업을 더 간편하게 작성할 수 있도록 도와주는 표현 언어예요.\n\n쉽게 말해, LCEL은 LangChain에서 다양한 데이터 처리, 조건문, 변수 활용 등을 간결하고 직관적으로 작성할 수 있게 해주는 도구라고 생각하시면 됩니다. 덕분에 사용자들은 코드 작성이 더 수월해지고, 복잡한 플로우도 쉽고 명확하게 표현할 수 있죠.\n\n혹시 LCEL을 직접 활용해보고 싶으시거나, 구체적인 사용법이나 예제가 궁금하시면 언제든 말씀해 주세요. 함께 천천히 살펴볼 수 있어요! 😊'